In [3]:
#不同算法的全局带宽总和对比
import pandas as pd

# 设置pandas选项，确保输出所有行和所有列，不折叠省略
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# H100_26H100_27H100_28H100_29   # Het-4Mix  #Part_mean_1111_bw_32dim_dynamicTrue.csv
df = pd.read_csv('./Data/Evaluation/H100_Real/H100_26H100_27H100_28H100_29/multi_tenant_simulation.csv') 
# 按需分组提取并展示全部内容
grouped = df.loc[df.groupby(['repeat_id','algorithm_name', 'search_mode'])['job_id'].idxmax()]
# grouped = grouped[grouped['algorithm_name'] != 'GroundTruth']

# 先输出原有字段，再新增 best_throughput 列
# 找到每个repeat_id下real_cluster_throughput最大值的那些algorithm_name
max_throughput = grouped.groupby('repeat_id')['real_cluster_throughput'].max()
# 找到每个repeat_id下达到最大throughput的行
max_rows = grouped[grouped.apply(lambda row: row['real_cluster_throughput'] == max_throughput[row['repeat_id']], axis=1)]
# 统计每个算法出现的次数
algo_counts = max_rows['algorithm_name'].value_counts()
print("各 repeat_id 下 real_cluster_throughput 达到最大值的算法及出现次数：")
for algo, count in algo_counts.items():
    print(f"  {algo}: {count} 次")
# 打印每个算法在哪些repeat_id下达到最大值
print("\n各算法达到最大值的 repeat_id：")
for algo in algo_counts.index:
    repeat_ids = max_rows[max_rows['algorithm_name'] == algo]['repeat_id'].tolist()
    print(f"  {algo}: {repeat_ids}")


各 repeat_id 下 real_cluster_throughput 达到最大值的算法及出现次数：
  UpperBandDisp: 82 次
  BandDisp: 61 次
  Topo: 37 次
  BandDisp_Greedy: 35 次
  Default: 29 次

各算法达到最大值的 repeat_id：
  UpperBandDisp: [0, 1, 3, 4, 5, 6, 8, 9, 10, 12, 13, 14, 15, 16, 17, 19, 20, 21, 24, 25, 26, 27, 28, 29, 30, 33, 34, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 58, 59, 61, 62, 63, 64, 65, 66, 67, 69, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 83, 84, 86, 87, 89, 90, 91, 92, 94, 95, 96, 97, 98]
  BandDisp: [0, 3, 4, 5, 9, 10, 14, 16, 17, 18, 19, 23, 26, 27, 28, 30, 31, 34, 37, 39, 40, 41, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 55, 57, 58, 60, 61, 62, 64, 65, 66, 71, 72, 73, 76, 78, 79, 80, 81, 82, 85, 86, 88, 90, 91, 94, 95, 96, 97, 98]
  Topo: [2, 7, 8, 9, 10, 19, 23, 27, 32, 33, 34, 35, 37, 41, 44, 48, 51, 57, 61, 63, 65, 66, 68, 70, 71, 72, 73, 75, 76, 77, 78, 81, 82, 85, 93, 94, 95]
  BandDisp_Greedy: [2, 6, 8, 9, 11, 17, 19, 22, 28, 32, 33, 36, 44, 55, 58, 59, 61, 65, 66

In [4]:
result = grouped[['repeat_id','job_id', 'algorithm_name', 'search_mode',  'real_cluster_throughput']].copy()
# 计算每个repeat_id下最大的real_cluster_throughput，并赋值给新列
best_throughput_map = grouped.groupby('repeat_id')['real_cluster_throughput'].max()
result['best_throughput'] = result['repeat_id'].map(best_throughput_map)
result["throughput_ratio"] = (result["real_cluster_throughput"] / result["best_throughput"]).round(2)

# 计算每个repeat_id下不同算法的real_final_bw的标准差
# 首先需要从原始df中获取每个repeat_id和algorithm_name对应的所有real_final_bw
bw_std_map = df.groupby(['repeat_id', 'algorithm_name'])['real_final_bw'].std().round(2).reset_index()
bw_std_map.columns = ['repeat_id', 'algorithm_name', 'real_final_bw_std']
# 将标准差合并到result中
result = result.merge(bw_std_map, on=['repeat_id', 'algorithm_name'], how='left')

# 筛选 repeat_id <= x 的数据
x = 100
filtered_result = result[result['repeat_id'] <= x]

# 按 repeat_id 分组显示，每组之间添加分割线
# for repeat_id in sorted(filtered_result['repeat_id'].unique()):
#     print(f"\n{'='*80}")
#     print(f"Repeat ID: {repeat_id}")
#     print(f"{'='*80}")
#     group_data = filtered_result[filtered_result['repeat_id'] == repeat_id]
#     print(group_data.to_string(index=False))

# 计算每个算法在repeat_id <= x 下的throughput_ratio总和
print(f'算法在repeat = {x} 下的throughput_ratio总和')
print(filtered_result.groupby('algorithm_name')['throughput_ratio'].sum().sort_values(ascending=False))
print("----"*20)
print(f'算法在repeat = {x} 下,多次workload分配的全局带宽的标准差的均值')
print(filtered_result.groupby('algorithm_name')['real_final_bw_std'].mean().sort_values(ascending=True))
# ===== 新增：算法在 repeat = x 下的总 throughput 的 ratio =====
print("----"*20)
print(f'算法在repeat = {x} 下的总throughput的ratio')
# 1. 每个算法在 repeat_id <= x 下的总 throughput
algo_total_throughput = filtered_result.groupby('algorithm_name')['real_cluster_throughput'].sum()
# 2. 找出最大的总 throughput 作为分母
max_total_throughput = algo_total_throughput.max()
# 3. 计算每个算法相对于最大总 throughput 的 ratio
algo_total_throughput_ratio = (algo_total_throughput / max_total_throughput).round(2).sort_values(ascending=False)
print(algo_total_throughput_ratio)

算法在repeat = 100 下的throughput_ratio总和
algorithm_name
UpperBandDisp      98.71
Topo               93.53
BandDisp           93.22
BandDisp_Greedy    92.41
Default            91.66
Random             32.29
Name: throughput_ratio, dtype: float64
--------------------------------------------------------------------------------
算法在repeat = 100 下,多次workload分配的全局带宽的标准差的均值
algorithm_name
Random             32.8376
UpperBandDisp      56.9201
BandDisp_Greedy    62.1577
BandDisp           62.9965
Topo               64.6478
Default            66.7948
Name: real_final_bw_std, dtype: float64
--------------------------------------------------------------------------------
算法在repeat = 100 下的总throughput的ratio
algorithm_name
UpperBandDisp      1.00
BandDisp           0.95
Topo               0.95
BandDisp_Greedy    0.93
Default            0.93
Random             0.32
Name: real_cluster_throughput, dtype: float64


In [48]:
df = pd.read_csv('./Data/Evaluation/H100_Real/H100_26H100_27H100_28H100_29/multi_tenant_simulation.csv')
result = df[(df['repeat_id'] == 9) & (df['algorithm_name'].isin(['GroundTruth','BandDisp','UpperBandDisp', 'Topo']))]
result = result.drop(columns=['search_mode'])
result.iloc[:, 1:]

FileNotFoundError: [Errno 2] No such file or directory: './Data/Evaluation/H100_Real/H100_26H100_27H100_28H100_29/multi_tenant_simulation.csv'